#### MetalLB

In [ ]:
kubectl apply -f https://raw.githubusercontent.com/metallb/metallb/v0.15.3/config/manifests/metallb-native.yaml

In [ ]:
kubectl delete validatingwebhookconfiguration metallb-webhook-configuration

Verfiy

In [ ]:
kubectl get pods -n metallb-system -w

Create pool

In [ ]:
vim metallb-pool.yaml

In [ ]:
apiVersion: metallb.io/v1beta1
kind: IPAddressPool
metadata:
  name: default-pool
  namespace: metallb-system
spec:
  addresses:
  - 172.16.6.90-172.16.6.95
---
apiVersion: metallb.io/v1beta1
kind: L2Advertisement
metadata:
  name: l2adv
  namespace: metallb-system

In [ ]:
kubectl apply -f metallb-pool.yaml

---

In [ ]:
kubectl edit configmap kube-proxy -n kube-system

In [ ]:
strictARP: true

In [ ]:
kubectl rollout restart ds kube-proxy -n kube-system

---

##### Ingress

In [ ]:
helm repo add ingress-nginx https://kubernetes.github.io/ingress-nginx
helm repo update

In [ ]:
vim nginx-ingress-values.yaml

In [ ]:
controller:
  replicaCount: 2
  hostNetwork: true
  dnsPolicy: ClusterFirstWithHostNet

  hostPort:
    enabled: true
    ports:
      http: 80
      https: 443

  service:
    type: LoadBalancer
    loadBalancerIP: 172.16.6.90

  admissionWebhooks:
    enabled: false

  metrics:
    enabled: true

  podDisruptionBudget:
    enabled: true


upgrade

In [ ]:
cat > nginx-ingress-values.yaml <<EOF
controller:
  kind: DaemonSet          # ← change from Deployment to DaemonSet
  hostNetwork: true
  dnsPolicy: ClusterFirstWithHostNet
  hostPort:
    enabled: true
    ports:
      http: 80
      https: 443
  service:
    type: LoadBalancer
    loadBalancerIP: 172.16.6.90
  admissionWebhooks:
    enabled: false
  metrics:
    enabled: true
  podDisruptionBudget:
    enabled: true
  tolerations:             # ← add this to run on master nodes
  - key: "node-role.kubernetes.io/control-plane"
    operator: "Exists"
    effect: "NoSchedule"
EOF

In [ ]:
helm upgrade --install ingress-nginx ingress-nginx/ingress-nginx \
  --namespace ingress-nginx \
  --create-namespace \
  -f nginx-ingress-values.yaml

In [ ]:
kubectl get svc -n ingress-nginx -w
kubectl rollout status deployment ingress-nginx-controller -n ingress-nginx

---

#### Headlamp

In [ ]:
# 1. Add the official Helm repo
helm repo add headlamp https://kubernetes-sigs.github.io/headlamp/
helm repo update

In [ ]:
vim headlamp-values.yaml

In [ ]:
ingress:
  enabled: true
  ingressClassName: nginx   
  hosts:
    - host: headlamp.voip.local
      paths:
        - path: /
          type: Prefix

In [ ]:
helm install my-headlamp headlamp/headlamp \
	-f headlamp-values.yaml \
	--namespace headlamp \
	--create-namespace

In [ ]:
  kubectl create token my-headlamp --namespace headlamp

In [ ]:
curl -H "Host: headlamp.voip.local" http://172.16.6.90